# Fermented Dairy Intake and Cardiometabolic Health: Dietary Data Preparation

**Student:** [Your Name]  
**Date:** March 2026  
**Dataset:** NHANES 2017–2018

---

## Abstract

This notebook prepares the dietary exposure data for an analysis of the association between fermented dairy consumption and cardiometabolic health outcomes in US adults, using the National Health and Nutrition Examination Survey (NHANES) 2017–2018 cycle. Two 24-hour dietary recall interviews are processed to classify each food item as dairy or fermented dairy, aggregate intake per participant, and average across the two recall days. The output is a single tidy dataset — `data/processed/nhanes_dietary_avg.csv` — containing each participant's average daily nutrient intake alongside their average dairy and fermented dairy consumption, ready for use in regression analyses.


## Setup

We begin by importing the libraries we need and defining the paths to our data files. Using `pathlib.Path` rather than plain strings means the paths work correctly on both Windows and macOS/Linux — Python handles the directory separator for us.

In [ ]:
import pandas as pd
import openpyxl
from pathlib import Path
import sys
import subprocess

# ── Detect environment and locate the project root ───────────────────────────
IN_COLAB = "google.colab" in sys.modules

REPO_NAME = "fb3pfb_nhanes"   # adjust if your repo is named differently

if IN_COLAB:
    repo_path = Path("/content") / REPO_NAME
    if not repo_path.exists():
        print("Cloning repository...")
        subprocess.run(
            ["git", "clone", f"https://github.com/ggkuhnle/{REPO_NAME}.git",
             str(repo_path)],
            check=True
        )
    ROOT = repo_path
else:
    # Local: notebook lives in notebooks/, root is one level up
    ROOT = Path.cwd().parent

RAW       = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Environment:             ", "Colab" if IN_COLAB else "local")
print("Root:                    ", ROOT)
print("Raw data directory:      ", RAW)
print("Processed data directory:", PROCESSED)


---

## Data Download

The four NHANES 2017–2018 dietary files are downloaded directly from the CDC public server 
if they are not already present in `data/raw/`. This means the notebook is fully self-contained — 
no manual file transfer is needed for these files.

| File | Contents |
|------|----------|
| `DR1IFF_J.xpt` | Day 1 individual foods |
| `DR2IFF_J.xpt` | Day 2 individual foods |
| `DR1TOT_J.xpt` | Day 1 total nutrient intakes |
| `DR2TOT_J.xpt` | Day 2 total nutrient intakes |


In [ ]:
import urllib.request

NHANES_BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles"

NHANES_FILES = [
    "DR1IFF_J.xpt",
    "DR2IFF_J.xpt",
    "DR1TOT_J.xpt",
    "DR2TOT_J.xpt",
]

for fname in NHANES_FILES:
    dest = RAW / fname
    if dest.exists():
        print(f"  already present — skipping: {fname}")
    else:
        url = f"{NHANES_BASE}/{fname}"
        print(f"  downloading {fname} ...", end=" ", flush=True)
        urllib.request.urlretrieve(url, dest)
        size_mb = dest.stat().st_size / 1_048_576
        print(f"done ({size_mb:.1f} MB)")

print("\nAll files ready.")


---

## Step 1 — Load the food classification lookup table

The USDA maintains a **Food and Nutrient Database for Dietary Studies (FNDDS)**, which assigns a unique integer *food code* to every food and beverage. Each NHANES dietary recall record references these codes, so we can link what a participant ate to a standardised food description.

For this analysis, we have extended the standard FNDDS "At A Glance" spreadsheet with two binary classification columns:

| Column | Meaning |
|--------|--------|
| `Dairy binary` | 1 if the food is a dairy product, 0 otherwise |
| `Fermented dairy binary` | 1 if the food is a *fermented* dairy product (e.g., yogurt, cheese, kefir), 0 otherwise |

The spreadsheet has a long title in row 1 and column headers in row 2; actual data begins in row 3. We skip the title row and read only the three columns we need: the food code (column index 0), the dairy flag (index 7), and the fermented dairy flag (index 8).

We use `openpyxl` with `data_only=True` because the file contains Excel formulas whose cached (pre-calculated) values we want — without this flag, openpyxl would return the formula text rather than the numeric result.

In [ ]:
fndds_path = PROCESSED / "2017-2018 FNDDS At A Glance - Foods and Beverages.xlsx"

wb = openpyxl.load_workbook(fndds_path, data_only=True)
ws = wb["Food and Beverages"]

# Row 1 = title, Row 2 = headers, Row 3 onwards = data.
# We iterate from row 3, extracting only the three columns we need.
rows = []
for row in ws.iter_rows(min_row=3, values_only=True):
    food_code = row[0]       # col index 0: USDA food code (integer)
    is_dairy = row[7]        # col index 7: dairy binary flag
    is_fermented = row[8]    # col index 8: fermented dairy binary flag
    if food_code is not None:  # skip any trailing empty rows
        rows.append({
            "food_code": int(food_code),
            "is_dairy": int(is_dairy) if is_dairy is not None else 0,
            "is_fermented_dairy": int(is_fermented) if is_fermented is not None else 0,
        })

food_lookup = pd.DataFrame(rows)

print("Lookup table shape:", food_lookup.shape)
print("Columns:", list(food_lookup.columns))
print(f"\nDairy foods: {food_lookup['is_dairy'].sum()} | Fermented dairy foods: {food_lookup['is_fermented_dairy'].sum()}")
food_lookup.head(10)

---

## Step 2 — Load the NHANES individual foods files

NHANES collects dietary information through **24-hour dietary recalls** — participants are interviewed by a trained nutritionist and asked to describe everything they ate and drank in the previous 24 hours. In the 2017–2018 cycle, most participants completed two such recalls: one in person (Day 1) and a follow-up by phone (Day 2).

The **individual foods files** (`DR1IFF_J` for Day 1, `DR2IFF_J` for Day 2) contain one row for each food item a participant reported — so a single participant will have as many rows as foods they described. The key variables are:

| Variable | Description |
|----------|------------|
| `SEQN` | Unique participant identifier |
| `DR1IFDCD` / `DR2IFDCD` | USDA food code (links to the FNDDS lookup) |
| `DR1IGRMS` / `DR2IGRMS` | Gram weight of the food item consumed |

The files are stored in SAS XPORT format (`.xpt`), which `pandas` can read directly.

In [ ]:
# Load Day 1 individual foods
iff_day1_raw = pd.read_sas(RAW / "DR1IFF_J.xpt", format="xport", encoding="utf-8")
print("Day 1 individual foods — shape:", iff_day1_raw.shape)
print("Columns:", list(iff_day1_raw.columns))
iff_day1_raw[["SEQN", "DR1IFDCD", "DR1IGRMS"]].head()

In [ ]:
# Load Day 2 individual foods
iff_day2_raw = pd.read_sas(RAW / "DR2IFF_J.xpt", format="xport", encoding="utf-8")
print("Day 2 individual foods — shape:", iff_day2_raw.shape)
print("Columns:", list(iff_day2_raw.columns))
iff_day2_raw[["SEQN", "DR2IFDCD", "DR2IGRMS"]].head()

In [ ]:
# Keep only the variables we need and standardise types
iff_day1 = iff_day1_raw[["SEQN", "DR1IFDCD", "DR1IGRMS"]].copy()
iff_day2 = iff_day2_raw[["SEQN", "DR2IFDCD", "DR2IGRMS"]].copy()

# SEQN and food codes come through as float64 from the XPT reader; convert to int
iff_day1["SEQN"] = iff_day1["SEQN"].astype(int)
iff_day1["DR1IFDCD"] = iff_day1["DR1IFDCD"].astype(int)

iff_day2["SEQN"] = iff_day2["SEQN"].astype(int)
iff_day2["DR2IFDCD"] = iff_day2["DR2IFDCD"].astype(int)

print("Day 1 unique participants:", iff_day1["SEQN"].nunique())
print("Day 2 unique participants:", iff_day2["SEQN"].nunique())

---

## Step 3 — Classify each food item as dairy / fermented dairy

Each row in the individual foods file contains a food code, but not yet a dairy flag. We need to bring in that information from the lookup table we built in Step 1. This is done with a **merge** (also called a join): we match rows from two tables based on a shared key column — in this case the food code.

We use a **left merge**, meaning we keep every row from the left-hand table (the individual foods file) regardless of whether a matching food code exists in the right-hand table (the lookup). This matters because:

- Not every food code in the dietary recall may appear in the FNDDS file (e.g. codes for combination foods or commercial products may differ slightly).
- We do not want to silently drop food records — we want to retain them and simply treat unmatched codes as non-dairy (flag = 0).

After merging, any row that did not find a match will have `NaN` in the flag columns; we fill those with 0.

In [ ]:
# --- Day 1 ---
iff_day1_classified = iff_day1.merge(
    food_lookup,
    left_on="DR1IFDCD",   # food code column in the Day 1 file
    right_on="food_code", # food code column in the lookup
    how="left",           # keep all Day 1 rows
)

# Fill NaN flags (unmatched food codes) with 0
iff_day1_classified["is_dairy"] = iff_day1_classified["is_dairy"].fillna(0).astype(int)
iff_day1_classified["is_fermented_dairy"] = iff_day1_classified["is_fermented_dairy"].fillna(0).astype(int)

# A left merge must not change the row count — assert this as a sanity check
assert len(iff_day1_classified) == len(iff_day1), (
    f"Row count changed after merge! Expected {len(iff_day1)}, got {len(iff_day1_classified)}. "
    "Check for duplicate food codes in the lookup table."
)

unmatched_day1 = iff_day1_classified["food_code"].isna().sum()
print(f"Day 1: {len(iff_day1_classified)} food records | {unmatched_day1} unmatched food codes (treated as non-dairy)")
print(f"Dairy records: {iff_day1_classified['is_dairy'].sum()} | Fermented dairy records: {iff_day1_classified['is_fermented_dairy'].sum()}")
iff_day1_classified.head()

In [ ]:
# --- Day 2 ---
iff_day2_classified = iff_day2.merge(
    food_lookup,
    left_on="DR2IFDCD",
    right_on="food_code",
    how="left",
)

iff_day2_classified["is_dairy"] = iff_day2_classified["is_dairy"].fillna(0).astype(int)
iff_day2_classified["is_fermented_dairy"] = iff_day2_classified["is_fermented_dairy"].fillna(0).astype(int)

assert len(iff_day2_classified) == len(iff_day2), (
    f"Row count changed after merge! Expected {len(iff_day2)}, got {len(iff_day2_classified)}. "
    "Check for duplicate food codes in the lookup table."
)

unmatched_day2 = iff_day2_classified["food_code"].isna().sum()
print(f"Day 2: {len(iff_day2_classified)} food records | {unmatched_day2} unmatched food codes (treated as non-dairy)")
print(f"Dairy records: {iff_day2_classified['is_dairy'].sum()} | Fermented dairy records: {iff_day2_classified['is_fermented_dairy'].sum()}")
iff_day2_classified.head()

---

## Step 4 — Summarise dairy intake per participant per day

Currently we have one row per *food item*. We want one row per *participant* per *day*, showing how many grams of dairy and fermented dairy they consumed in total.

The logic is:
1. Filter to rows where `is_dairy == 1` and sum the gram weights, grouped by participant (`SEQN`). This gives `total_dairy_g`.
2. Repeat for `is_fermented_dairy == 1` to get `fermented_dairy_g`.
3. Combine these two series into a single summary DataFrame.

**Handling zero consumers:** Participants who ate no dairy on a given day will not appear in the filtered subsets at all. We use `.reindex()` to re-insert every participant from the full individual foods file, assigning 0 grams where they are absent. This is important — we must not drop non-dairy consumers from the dataset.

In [ ]:
def summarise_dairy_intake(classified_df, gram_col):
    """
    Compute total dairy and fermented dairy grams per participant.

    Parameters
    ----------
    classified_df : DataFrame with columns SEQN, <gram_col>, is_dairy, is_fermented_dairy
    gram_col      : name of the column holding gram weights (e.g. 'DR1IGRMS')

    Returns
    -------
    DataFrame with columns: SEQN, total_dairy_g, fermented_dairy_g
    """
    all_seqns = classified_df["SEQN"].unique()

    # Sum gram weights of dairy foods per participant
    dairy_g = (
        classified_df.loc[classified_df["is_dairy"] == 1]
        .groupby("SEQN")[gram_col]
        .sum()
        .reindex(all_seqns, fill_value=0)  # ensure non-consumers appear with 0
        .rename("total_dairy_g")
    )

    # Sum gram weights of fermented dairy foods per participant
    fermented_g = (
        classified_df.loc[classified_df["is_fermented_dairy"] == 1]
        .groupby("SEQN")[gram_col]
        .sum()
        .reindex(all_seqns, fill_value=0)
        .rename("fermented_dairy_g")
    )

    # Combine into a single DataFrame
    summary = pd.concat([dairy_g, fermented_g], axis=1).reset_index()
    summary.rename(columns={"index": "SEQN"}, inplace=True)
    return summary


dairy_day1 = summarise_dairy_intake(iff_day1_classified, "DR1IGRMS")
dairy_day2 = summarise_dairy_intake(iff_day2_classified, "DR2IGRMS")

print("Day 1 dairy summary — shape:", dairy_day1.shape)
print(dairy_day1.describe())
dairy_day1.head()

In [ ]:
print("Day 2 dairy summary — shape:", dairy_day2.shape)
print(dairy_day2.describe())
dairy_day2.head()

---

## Step 5 — Load the total nutrient intake files and average across two days

In addition to the food-level files, NHANES provides **total nutrient intake files** (`DR1TOT_J` and `DR2TOT_J`) that contain one row per participant per day, with pre-calculated totals for energy, macronutrients, vitamins, minerals, and so on. These are the variables we will use as covariates in the analysis.

**Why average across two days rather than using just one?**  
A single 24-hour recall captures what a person happened to eat on one particular day, which may be quite atypical for that individual — this is called *within-person day-to-day variability*. Averaging across two recall days reduces this random noise and gives a better estimate of *habitual* intake. This is standard practice in dietary epidemiology.

In [ ]:
# Load the two total nutrient files
tot_day1 = pd.read_sas(RAW / "DR1TOT_J.xpt", format="xport", encoding="utf-8")
tot_day2 = pd.read_sas(RAW / "DR2TOT_J.xpt", format="xport", encoding="utf-8")

print("DR1TOT shape:", tot_day1.shape, "| Columns:", list(tot_day1.columns[:10]), "...")
print("DR2TOT shape:", tot_day2.shape)
tot_day1.head(3)

In [ ]:
# Stack the two days into one long DataFrame, then average by participant
# pd.concat with axis=0 stacks rows; ignore_index resets the row numbers
tot_both_days = pd.concat([tot_day1, tot_day2], axis=0, ignore_index=True)
print("Combined (long) shape:", tot_both_days.shape)

# Convert SEQN to int so it matches our other DataFrames
tot_both_days["SEQN"] = tot_both_days["SEQN"].astype(int)

# Average all numeric columns across the two days for each participant
# numeric_only=True skips any non-numeric columns that might exist
nutrients_avg = tot_both_days.groupby("SEQN").mean(numeric_only=True).reset_index()

print("Averaged dataset shape:", nutrients_avg.shape)
print(f"Unique participants: {nutrients_avg['SEQN'].nunique()}")
nutrients_avg.head(3)

---

## Step 6 — Add dairy variables to the averaged nutrient dataset

We now have two pieces of information for each participant:

1. **`nutrients_avg`** — average daily nutrient intakes across two recall days.
2. **`dairy_day1` / `dairy_day2`** — dairy and fermented dairy grams for each individual recall day.

We calculate each participant's average dairy intake across the two days, then merge that into `nutrients_avg`. Participants who appear in the nutrient totals file but have no individual food records (e.g. due to data quality exclusions) are assigned 0 grams of dairy.

In [ ]:
# Combine the two per-day dairy summaries and average across days
dairy_both_days = pd.concat(
    [dairy_day1.assign(recall_day=1), dairy_day2.assign(recall_day=2)],
    axis=0,
    ignore_index=True,
)

dairy_avg = (
    dairy_both_days
    .groupby("SEQN")[["total_dairy_g", "fermented_dairy_g"]]
    .mean()
    .reset_index()
    .rename(columns={
        "total_dairy_g": "avg_total_dairy_g",
        "fermented_dairy_g": "avg_fermented_dairy_g",
    })
)

print("Dairy averages shape:", dairy_avg.shape)
print(dairy_avg.describe())
dairy_avg.head()

In [ ]:
# Merge the dairy averages into the nutrient averages
# left merge: keep all participants from nutrients_avg
# Participants absent from dairy_avg (no food-level records) get NaN → fill with 0
nutrients_with_dairy = nutrients_avg.merge(dairy_avg, on="SEQN", how="left")

nutrients_with_dairy["avg_total_dairy_g"] = nutrients_with_dairy["avg_total_dairy_g"].fillna(0)
nutrients_with_dairy["avg_fermented_dairy_g"] = nutrients_with_dairy["avg_fermented_dairy_g"].fillna(0)

# Sanity check: row count must not change
assert len(nutrients_with_dairy) == len(nutrients_avg), (
    f"Unexpected row count after merge: {len(nutrients_with_dairy)} vs {len(nutrients_avg)}. "
    "The dairy_avg table may contain duplicate SEQNs."
)

print("Final shape:", nutrients_with_dairy.shape)
print("\nDairy variable summary:")
print(nutrients_with_dairy[["avg_total_dairy_g", "avg_fermented_dairy_g"]].describe())
nutrients_with_dairy.head()

---

## Step 7 — Save the final dataset

The prepared dataset is saved as a CSV file to `data/processed/nhanes_dietary_avg.csv`. CSV (comma-separated values) is a plain-text format that can be opened in Excel, R, or any other analysis tool without needing special software.

We set `index=False` so that pandas does not write its internal row numbers as an extra column — the data already has a proper participant identifier (`SEQN`).

**What this file contains:** One row per NHANES 2017–2018 participant who completed at least one dietary recall. Each row holds their average daily nutrient intakes (averaged across the two recall days), plus two new exposure variables: `avg_total_dairy_g` (average grams of all dairy consumed per day) and `avg_fermented_dairy_g` (average grams of fermented dairy consumed per day).

**How to use it:** This file should be the starting point for subsequent notebooks that merge in health outcome data (e.g. blood pressure, cholesterol, BMI from other NHANES files) and run regression models.

In [ ]:
output_path = PROCESSED / "nhanes_dietary_avg.csv"
nutrients_with_dairy.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Shape: {nutrients_with_dairy.shape}")
print("\nFirst 5 rows:")
nutrients_with_dairy.head(5)

---

## Summary

This notebook has produced:

**Output file:** `data/processed/nhanes_dietary_avg.csv`

**Key variables:**

| Variable | Description |
|----------|------------|
| `SEQN` | NHANES participant identifier |
| `avg_total_dairy_g` | Average grams of total dairy consumed per day (mean of Day 1 and Day 2) |
| `avg_fermented_dairy_g` | Average grams of fermented dairy consumed per day (mean of Day 1 and Day 2) |
| All `DR1`/`DR2` nutrient variables | Averaged across two recall days (energy, macronutrients, micronutrients) |

**Processing steps completed:**
1. Food classification lookup table loaded from FNDDS 2017–2018 (Excel).
2. NHANES individual foods files (Days 1 and 2) loaded and merged with the lookup.
3. Dairy and fermented dairy grams aggregated per participant per day.
4. Total nutrient intake files loaded and averaged across two recall days.
5. Dairy exposure variables merged into the nutrient dataset and saved.

**Next step:** Load `nhanes_dietary_avg.csv` in the next notebook alongside the NHANES demographic and health outcome files (e.g. `DEMO_J.xpt`, `BPX_J.xpt`, `TRIGLY_J.xpt`) to build the analytical dataset for regression modelling.